In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import time
import os
import json
import ast
import sys
import tqdm
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
def obs_to_fragment_man(nodes, configs):
    fragment_map = np.zeros((30,30))
    for ii, node in enumerate(nodes):
        fragment_map[node[0]-2+3:node[0]+3+3,node[1]-2+3:node[1]+3+3] += 1*configs[ii]
    return np.minimum(1,fragment_map[3:27,3:27])

In [ ]:
files = os.listdir(os.getcwd()+ "/data/raw_data")

In [ ]:
all_data = []
targets = []
sample_id = 0
for episode in range(1000):
    with open(f"data/raw_data/{files[episode]}") as f:
        data = json.load(f)
    params = data["steps"][0][0]["info"]["replay"]["params"]
    sap_range = params["unit_sap_range"]
    for step in range(505):
        energy_map = np.array(data["steps"][step][0]["info"]["replay"]["observations"][0]["map_features"]["energy"])
        tile_map = np.array(data["steps"][step][0]["info"]["replay"]["observations"][0]["map_features"]["tile_type"])
        energy_map[tile_map==1] -= params["nebula_tile_energy_reduction"]
        energy_map = (energy_map-energy_map.mean())/(energy_map.std()+1e-8)
        tile_map[tile_map==1] = 0
        tile_map[tile_map==2] = 1
        
        nodes = data["steps"][step][0]["info"]["replay"]["observations"][0]["relic_nodes"]
        configs = data["steps"][step][0]["info"]["replay"]["observations"][0]["relic_node_configs"]
        fragment_map = obs_to_fragment_man(nodes, configs)
        player = 0
        obs = json.loads(data["steps"][step][player]["observation"]["obs"])
        positions = obs["units"]["position"]
        masks = [obs["units_mask"]]
        own_ids = np.arange(16)[masks[0][0]]
        own_positions = np.array(positions[0])[masks[0][0]]
        own_positions_map = np.zeros((24,24))
        own_positions_map[own_positions[:,0],own_positions[:,1]] = 1
        enemy_positions = np.array(positions[1])[masks[0][1]]
        enemy_positions_map = np.zeros((24,24))
        enemy_positions_map[enemy_positions[:,0],enemy_positions[:,1]] = 1
        actions = np.array(data["steps"][step+1][player]["action"])
        if enemy_positions.size!=0:
            for ii, pos in enumerate(own_positions):
                if np.max(np.abs((enemy_positions-pos)))<=8: # valid datapoint if in max sap_range
                    in_pos_map = np.zeros((24,24))
                    in_pos_map[pos[0],pos[1]] = 1
                    in_own_positions = own_positions_map.copy()
                    in_own_positions[pos[0],pos[1]] = 0
                    in_enemy_positions = enemy_positions_map.copy()
                    in_fragment_map = fragment_map.copy()
                    X = np.concatenate((np.expand_dims(in_pos_map,axis=0), np.expand_dims(in_own_positions,axis=0), np.expand_dims(in_enemy_positions,axis=0), 
                                        np.expand_dims(in_fragment_map,axis=0), np.expand_dims(tile_map,axis=0), np.expand_dims(energy_map,axis=0)),axis=0)
                    y = actions[ii]
                    sample = (torch.tensor(X),torch.tensor(y))
                    torch.save(sample, f"data/extracted_data/id_{sample_id}")
                    sample_id += 1

In [ ]:
class PlaysDataset(torch.utils.data.Dataset):
    def __init__(self, root, num=1e4):
        self.num = int(num)
        self.data = []
        self.files = os.listdir(root)
        for i in range(int(num)):
            self.data.append(torch.load(os.path.join(root, self.files[i]))) # take all files in the root directory
    def __len__(self):
        return self.num
    def __getitem__(self, idx):
        sample, label = self.data[idx] # load the features of this sample
        return sample, label

In [ ]:
traindata = PlaysDataset(root="data/extracted_data", num=1e5)

In [ ]:
class ActionOracle(torch.nn.Module):
    def __init__(self,n_maps):
        super().__init__()
        self.n_maps = n_maps
        self.cnn = nn.Sequential(
                nn.Conv2d(self.n_maps, 16, kernel_size=1, padding=1),
                nn.ReLU(),
                #nn.MaxPool2d(2),
                nn.Conv2d(16, 16, kernel_size=5, padding=1),
                nn.ReLU(),
                nn.Conv2d(16, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(16, 8, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(8, 4, kernel_size=3, padding=1),
                nn.ReLU(),
                #nn.AvgPool2d(2),
                nn.Flatten(),
                nn.Linear(24*24*4, 1024),
                nn.ReLU(),
                nn.Linear(1024,128),
                nn.ReLU(),
                nn.Linear(128, 6),
                nn.Softmax(),
            )
    def forward(self, x):
        return self.cnn(x)

In [23]:
trainloader = torch.utils.data.DataLoader(traindata, batch_size=256, shuffle=True)
counts = torch.zeros((6))
for X,y in trainloader:
    count = torch.unique(y[:,0], return_counts=True)[1]
    counts +=count

In [ ]:
batch_size = 256
trainloader = torch.utils.data.DataLoader(traindata, batch_size=256, shuffle=True)
train_len = len(traindata)
lr = 1e-2
n_maps = 6
t = 0
model = ActionOracle(n_maps)
criterion = nn.CrossEntropyLoss(weight=1/counts)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
for epoch in range(100):
    losses = []
    accuracies= []
    n_ac = []
    for batch_num, (X, y) in enumerate(tqdm.tqdm(trainloader)):
        X, y = X.to(torch.float32), y[:,0]
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        losses.append(loss.item())
        pred = torch.argmax(out.detach(), dim=1)
        n_ac.append(torch.max(torch.unique(pred, return_counts=True)[1]).to(torch.float32))
        uniques = torch.unique(pred, return_counts=True)
        accuracy = (1*(pred==y)).to(torch.float32).mean()
        accuracies.append(accuracy)
        loss.backward()
        optimizer.step()
    print(torch.mean(torch.tensor(losses)))
    print(torch.mean(torch.tensor(accuracies)))
    print(torch.mean(torch.tensor(n_ac)/batch_size))

100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:37<00:00, 10.31it/s]


tensor(1.7920)
tensor(0.2101)
tensor(0.9990)


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:33<00:00, 11.73it/s]


tensor(1.7919)
tensor(0.1638)
tensor(0.9990)


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:44<00:00,  8.84it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:32<00:00, 12.09it/s]


tensor(1.7919)
tensor(0.2767)
tensor(0.9990)


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:31<00:00, 12.43it/s]


tensor(1.7918)
tensor(0.2440)
tensor(0.9990)


 64%|███████████████████████████████████████████████████▏                            | 250/391 [00:20<00:13, 10.57it/s]